In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

API_KEY = "286a083bfeaa68b3964f38b5f144a1554b7e6271af010f4724609adecd4ede08"
BASE_URL = "https://api.openaq.org/v3"
LAT, LON = "40.7128", "-74.0060"
YEAR = 2025
HEADERS = {"X-API-Key": API_KEY}
response = requests.get(
    f"{BASE_URL}/locations",
    headers=HEADERS,
    params={
        "coordinates": f"{LAT},{LON}",
        "radius": "25000"
    },
    timeout=30
)

locations = response.json().get("results", [])
target_sensors = []
for location in locations:
    for sensor in location.get("sensors", []):
        parameter = sensor.get("parameter", {}).get("name")
        if parameter in ["pm25", "no2"]:
            target_sensors.append({
                "sensor_id": sensor["id"],
                "parameter": parameter,
                "location_id": location["id"],
                "latitude": location["coordinates"]["latitude"],
                "longitude": location["coordinates"]["longitude"]
            })

all_measurements = []
for sensor in target_sensors:
    for month in range(1, 13):
        start_date = datetime(YEAR, month, 1)
        if month == 12:
            end_date = datetime(YEAR + 1, 1, 1)
        else:
            end_date = datetime(YEAR, month + 1, 1)
        page = 1

        while True:
            params = {
                "datetime_from": start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "datetime_to": end_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "page": page,
                "limit": 1000
            }
            try:
                response = requests.get(
                    f"{BASE_URL}/sensors/{sensor['sensor_id']}/measurements",
                    headers=HEADERS,
                    params=params,
                    timeout=30
                )

                if response.status_code == 429:
                    time.sleep(10)
                    continue

                response.raise_for_status()
                measurements = response.json().get("results", [])
                if not measurements:
                    break

                for measurement in measurements:
                    all_measurements.append({
                        "sensor_id": sensor["sensor_id"],
                        "parameter": sensor["parameter"],
                        "location_id": sensor["location_id"],
                        "latitude": sensor["latitude"],
                        "longitude": sensor["longitude"],
                        "datetime_utc": measurement["period"]["datetimeFrom"]["utc"],
                        "value": measurement["value"]
                    })
                page += 1

            except Exception as error:
                print(f"Error: {error}. Waiting 10 seconds...")
                time.sleep(10)
                break
        time.sleep(1.5)

print(f"Total measurements fetched: {len(all_measurements)}")
if all_measurements:
    pandas_df = pd.DataFrame(all_measurements)
    spark_df = spark.createDataFrame(pandas_df)
    (spark_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("Bronze_Lakehouse.dbo.openaq_raw")
    )